# Experiment 5: Subword-Level Tokenization, POS Tagging and Word2Vec

**Platform:** Google Colab  
**Subject:** Natural Language Processing (NLP)

## Aim
To implement subword-level tokenization using BPE and SentencePiece (with pretrained and locally trained tokenizers), perform Part-of-Speech (POS) tagging using spaCy and NLTK, and train Word2Vec CBOW and Skip-Gram models to explore word vectors and similarities.

## Software / Libraries
Python, Hugging Face Transformers, Tokenizers, SentencePiece, spaCy, NLTK, Gensim, NumPy, and pandas.

## Instructions
1. Open this notebook in Google Colab.
2. Select **Runtime → Run all**. The first run installs packages and downloads pretrained models, so it may take a few minutes.
3. For the “without pretrained model” tasks, the tokenizer is trained on the experiment's supplied corpus.
4. Word2Vec results can vary slightly because the models are trained on a very small corpus.

## Input Data

The following text is from the supplied `input_sub_word_data.txt` file. It is embedded here so the notebook can run in Colab without requiring a separate upload. You may replace it with the contents of your file if your instructor provides an updated corpus.

In [ ]:
input_text = """Natural language processing is a branch of artificial intelligence that
enables computers to understand, interpret, and generate human language.
Natural language processing combines linguistics, computer science, and
machine learning to process large amounts of natural language data.

Tokenization is one of the fundamental steps in natural language processing.
A tokenizer divides text into smaller units called tokens. Tokens may be
words, characters, or subword units. Word-level tokenization is simple and
easy to understand, but it may have difficulty handling rare and unknown
words.

Subword-level tokenization provides an effective solution to the problem
of rare words. Instead of representing every complete word as a separate
token, a subword tokenizer represents a word using smaller meaningful units.
For example, the word playing can be represented as play and ing. Similarly,
the word unhappy can be represented as un and happy.

Byte Pair Encoding is a popular subword tokenization algorithm. BPE starts
with a vocabulary containing individual characters and repeatedly merges
the most frequent pair of adjacent symbols. The merging process continues
until the desired vocabulary size is obtained. Frequent character sequences
gradually become larger subword units.

BPE is useful because it provides a balance between word-level and
character-level tokenization. Common words can be represented as complete
tokens, while rare words can be divided into smaller subword units. This
allows the tokenizer to represent words that were not frequently observed
during training.

Machine learning algorithms require numerical representations of text.
After tokenization, each token can be assigned a unique numerical token ID.
These token IDs can then be converted into vectors and supplied to machine
learning or deep learning models.

Deep learning has significantly improved natural language processing.
Neural networks can learn complex patterns from large collections of text.
Recurrent neural networks, long short-term memory networks, convolutional
neural networks, and transformer models have all been used for language
processing tasks.

Transformers use attention mechanisms to identify relationships between
different tokens in a sequence. The attention mechanism allows a model to
consider important words even when they occur far apart in a sentence.
Transformer-based models have become widely used for text classification,
machine translation, question answering, summarization, and text generation.

Language models learn statistical relationships between tokens. A language
model can predict the next token based on the tokens that appear before it.
Modern language models are trained on very large text corpora containing
books, articles, websites, documents, and other forms of written language.

Consider the word internationalization. A word-level tokenizer may treat
internationalization as one token. A subword tokenizer can divide it into
smaller units such as international, ization, or other learned subword
segments. The exact segmentation depends on the vocabulary and the BPE
merge rules learned from the training corpus.

Similarly, words such as computational, communication, representation,
classification, information, educational, environmental, technological,
development, understanding, processing, and programming contain recurring
subword patterns. BPE can learn these recurring patterns from the corpus.

For example, the suffixes ing, ed, tion, ment, ness, and able may occur
frequently in a corpus. BPE can learn frequent sequences containing these
characters and use them as subword tokens. Prefixes such as un, re, pre,
dis, and con may also appear frequently and can become useful subword units.

A larger corpus generally produces more useful BPE vocabulary because the
algorithm has more examples from which to learn frequent character pairs.
A small corpus may produce vocabulary that is specific to the sentences
used for training. Therefore, a representative corpus is important when
building a practical tokenizer.

In an NLP experiment, the corpus can be converted to lowercase and divided
into words before learning BPE merges. Each word can initially be represented
as a sequence of characters followed by an end-of-word marker. The algorithm
then counts adjacent symbol pairs and identifies the most frequent pair.

The most frequent pair is merged into a new symbol. After every merge, the
pair frequencies are recalculated. This process is repeated several times.
The resulting collection of symbols forms the learned subword vocabulary.

Suppose a corpus contains many occurrences of the sequence t and h.
The BPE algorithm may first learn the pair th. If the sequence th occurs
frequently with e, the algorithm may subsequently learn the sequence the.
With additional training data, longer frequently occurring sequences can be
learned.

BPE is therefore a data-driven tokenization technique. It does not require
a manually constructed list of prefixes, suffixes, or linguistic rules.
Instead, the vocabulary is learned automatically from the statistical
properties of the training corpus.

Subword tokenization is particularly useful for multilingual applications.
Different languages may have different word formation patterns, and a
word-level vocabulary can become extremely large. Subword units can reduce
the vocabulary size while still allowing the tokenizer to represent many
different words.

Text preprocessing can include converting characters to lowercase,
removing unnecessary spaces, normalizing punctuation, and separating
sentences. However, preprocessing decisions should be selected according
to the objective of the NLP application.

A BPE tokenizer can also be evaluated by examining the number of unique
tokens, vocabulary size, number of merges, average tokens per word, and
frequency of individual subword units. These measurements help students
understand how the tokenizer behaves as the number of BPE merges changes.

In this laboratory experiment, the above corpus can be used as training
data for implementing BPE without using a pretrained tokenizer. Students
can experiment with different numbers of merges and compare the resulting
subword vocabularies.

The tokenizer can then be tested on new words such as naturalization,
internationalization, computationally, unbelievable, preprocessing,
classification, communication, renewable, hydrokinetic, multilingual,
understanding, and modernization.

The purpose of this experiment is to understand how BPE learns reusable
subword units and how these units can be converted into token IDs for use
in natural language processing systems."""

print("Input text loaded.")
print("Number of characters:", len(input_text))
print("Preview:", input_text[:250], "...")

# 5.1 BPE-Based Subword Tokenization

**Question:** Implement subword-level tokenization using a BPE-based tokenizer. Display the generated tokens and their token IDs using (a) a pretrained model and (b) a tokenizer trained without a pretrained model.

### 5.1(a) BPE tokenizer using a pretrained model

In [ ]:
!pip -q install transformers tokenizers

In [ ]:
from transformers import AutoTokenizer

# GPT-2 uses byte-level BPE tokenization.
bpe_pretrained = AutoTokenizer.from_pretrained("gpt2")

encoded = bpe_pretrained(input_text, add_special_tokens=False)
tokens = bpe_pretrained.convert_ids_to_tokens(encoded["input_ids"])

print("First 80 tokens and their token IDs:")
for token, token_id in list(zip(tokens, encoded["input_ids"]))[:80]:
    print(f"Token: {token!r:25} Token ID: {token_id}")

print("\nTotal tokens:", len(tokens))
print("Vocabulary size:", bpe_pretrained.vocab_size)

### 5.1(b) BPE tokenizer trained from the supplied corpus (no pretrained tokenizer)

In [ ]:
from tokenizers import ByteLevelBPETokenizer
from pathlib import Path

# Train a new BPE vocabulary from the provided text.
training_file = "/content/bpe_training_corpus.txt"
Path(training_file).write_text(input_text, encoding="utf-8")

bpe_custom = ByteLevelBPETokenizer()
bpe_custom.train(
    files=[training_file],
    vocab_size=1000,
    min_frequency=2,
    special_tokens=["<s>", "<pad>", "</s>", "<unk>", "<mask>"]
)

custom_encoding = bpe_custom.encode(input_text)
print("First 80 tokens and their token IDs:")
for token, token_id in list(zip(custom_encoding.tokens, custom_encoding.ids))[:80]:
    print(f"Token: {token!r:25} Token ID: {token_id}")

print("\nTotal tokens:", len(custom_encoding.tokens))
print("Learned vocabulary size:", bpe_custom.get_vocab_size())

**Observation:** The pretrained GPT-2 tokenizer uses a vocabulary and merge rules learned before this experiment. The custom BPE tokenizer learns its vocabulary from the supplied text, so its token boundaries and IDs will generally differ.

# 5.2 SentencePiece Subword Tokenization

**Question:** Implement SentencePiece tokenization on the same input data, displaying tokens and token IDs using (a) a pretrained model and (b) a SentencePiece model trained on the supplied corpus.

### 5.2(a) SentencePiece tokenizer using a pretrained model

In [ ]:
!pip -q install sentencepiece transformers

In [ ]:
from transformers import AutoTokenizer

# T5's tokenizer is SentencePiece-based.
sp_pretrained = AutoTokenizer.from_pretrained("google-t5/t5-small")

sp_encoded = sp_pretrained(input_text, add_special_tokens=False)
sp_tokens = sp_pretrained.convert_ids_to_tokens(sp_encoded["input_ids"])

print("First 80 SentencePiece tokens and their token IDs:")
for token, token_id in list(zip(sp_tokens, sp_encoded["input_ids"]))[:80]:
    print(f"Token: {token!r:25} Token ID: {token_id}")

print("\nTotal tokens:", len(sp_tokens))
print("Vocabulary size:", sp_pretrained.vocab_size)

### 5.2(b) SentencePiece tokenizer trained from the supplied corpus (no pretrained model)

In [ ]:
import sentencepiece as spm
from pathlib import Path

sp_corpus_path = "/content/sentencepiece_training_corpus.txt"
sp_model_prefix = "/content/custom_sentencepiece"
Path(sp_corpus_path).write_text(input_text, encoding="utf-8")

# SentencePiece requires vocab_size to be feasible for the training corpus.
spm.SentencePieceTrainer.train(
    input=sp_corpus_path,
    model_prefix=sp_model_prefix,
    vocab_size=500,
    model_type="bpe",
    character_coverage=1.0,
    hard_vocab_limit=False,
    pad_id=0,
    unk_id=1,
    bos_id=2,
    eos_id=3
)

sp_custom = spm.SentencePieceProcessor(model_file=sp_model_prefix + ".model")
custom_sp_ids = sp_custom.encode(input_text, out_type=int)
custom_sp_tokens = sp_custom.encode(input_text, out_type=str)

print("First 80 SentencePiece tokens and their token IDs:")
for token, token_id in list(zip(custom_sp_tokens, custom_sp_ids))[:80]:
    print(f"Token: {token!r:25} Token ID: {token_id}")

print("\nTotal tokens:", len(custom_sp_tokens))
print("Learned vocabulary size:", sp_custom.get_piece_size())

**Observation:** The pretrained SentencePiece tokenizer uses the vocabulary of the pretrained T5 model. The locally trained SentencePiece model learns pieces from the supplied corpus. Token IDs are specific to each tokenizer and should not be compared across different vocabularies as if they had the same meaning.

# 5.3 Part-of-Speech (POS) Tagging Using spaCy and NLTK

**Input sentence:**  
“The young student is reading an interesting book in the library.”

**Question:** Display each token, its POS tag, and the description of the POS tag using (a) spaCy and (b) NLTK.

### 5.3(a) POS tagging using spaCy

In [ ]:
!pip -q install spacy
!python -m spacy download en_core_web_sm -q

In [ ]:
import spacy
import pandas as pd

nlp = spacy.load("en_core_web_sm")
sentence = "The young student is reading an interesting book in the library."
doc = nlp(sentence)

spacy_pos_rows = []
for token in doc:
    if not token.is_space:
        spacy_pos_rows.append({
            "Token": token.text,
            "POS Tag": token.pos_,
            "POS Description": spacy.explain(token.pos_) or "No description available"
        })

spacy_pos_df = pd.DataFrame(spacy_pos_rows)
display(spacy_pos_df)

### 5.3(b) POS tagging using NLTK

In [ ]:
!pip -q install nltk

In [ ]:
import nltk
nltk.download("averaged_perceptron_tagger_eng", quiet=True)
nltk.download("punkt_tab", quiet=True)
nltk.download("universal_tagset", quiet=True)

from nltk import word_tokenize, pos_tag
from nltk.help import upenn_tagset

sentence = "The young student is reading an interesting book in the library."
nltk_tokens = word_tokenize(sentence)
nltk_tagged = pos_tag(nltk_tokens)

# Penn Treebank tag descriptions for the tags in this sentence.
tag_descriptions = {
    "DT": "Determiner",
    "JJ": "Adjective",
    "NN": "Noun, singular or mass",
    "NNS": "Noun, plural",
    "VB": "Verb, base form",
    "VBD": "Verb, past tense",
    "VBG": "Verb, gerund or present participle",
    "VBN": "Verb, past participle",
    "VBP": "Verb, non-3rd-person singular present",
    "VBZ": "Verb, 3rd-person singular present",
    "IN": "Preposition or subordinating conjunction",
    ".": "Sentence-final punctuation",
    ",": "Comma"
}

nltk_pos_df = pd.DataFrame([
    {
        "Token": token,
        "POS Tag": tag,
        "POS Description": tag_descriptions.get(tag, "Penn Treebank tag")
    }
    for token, tag in nltk_tagged
])
display(nltk_pos_df)

**Observation:** POS taggers assign grammatical categories based on context. spaCy uses its own tag set (including universal POS labels), while NLTK's `pos_tag` returns Penn Treebank tags. Therefore, the tag labels and descriptions may differ between the two libraries.

# 5.4 POS Tagging, Unique Tokens, Descriptions and Frequencies Using spaCy

**Question:** Apply spaCy POS tagging to the supplied input text and display the unique token, POS tag, POS description, and frequency of each token.

In [ ]:
# Reuse the spaCy model loaded in Section 5.3.
doc = nlp(input_text)

from collections import Counter, defaultdict

# Count occurrences by (lowercased token, POS tag) to keep grammatical uses distinct.
token_counts = Counter()
token_descriptions = {}

for token in doc:
    if token.is_space or token.is_punct:
        continue
    key = (token.text.lower(), token.pos_)
    token_counts[key] += 1
    token_descriptions[key] = spacy.explain(token.pos_) or "No description available"

pos_frequency_df = pd.DataFrame([
    {
        "Unique Token": token,
        "POS Tag": pos,
        "POS Description": token_descriptions[(token, pos)],
        "Frequency": count
    }
    for (token, pos), count in token_counts.most_common()
])

display(pos_frequency_df)
print("Number of unique token/POS pairs:", len(pos_frequency_df))

**Observation:** Tokens are converted to lowercase for frequency counting, so “Natural” and “natural” are counted together. Punctuation and whitespace are excluded. A token appearing with different POS tags is shown as a separate token/POS pair.

# 5.5 Word2Vec CBOW Model

**Question:** Train a Word2Vec Continuous Bag of Words (CBOW) model using the given corpus. Display the vector representation of a selected word and find the three most similar words.

## Corpus

In [ ]:
corpus = [
    "Natural language processing is interesting",
    "Natural language processing is useful",
    "I love machine learning",
    "Machine learning is useful",
    "Deep learning is a part of machine learning",
    "Artificial intelligence is changing the world",
    "Natural language processing uses machine learning",
    "Deep learning and artificial intelligence are related",
    "I study natural language processing",
    "Machine learning helps artificial intelligence"
]

print("Number of sentences:", len(corpus))
for line in corpus:
    print("-", line)

In [ ]:
!pip -q install gensim

In [ ]:
from gensim.models import Word2Vec
from nltk.tokenize import word_tokenize
import nltk

nltk.download("punkt", quiet=True)
nltk.download("punkt_tab", quiet=True)

tokenized_corpus = [word_tokenize(line.lower()) for line in corpus]

# CBOW: sg=0
cbow_model = Word2Vec(
    sentences=tokenized_corpus,
    vector_size=50,
    window=3,
    min_count=1,
    workers=1,
    sg=0,
    seed=42,
    epochs=100
)

selected_word = "machine"
print(f"Vector representation of '{selected_word}':")
print(cbow_model.wv[selected_word])

print("\nThree most similar words:")
for word, similarity in cbow_model.wv.most_similar(selected_word, topn=3):
    print(f"{word}: {similarity:.4f}")

**Observation:** CBOW predicts a target word from its surrounding context. The displayed vector is a learned numerical representation. Because this corpus is small, similarity results are for demonstration and should not be treated as reliable semantic measurements.

# 5.6 Word2Vec Skip-Gram Model

**Question:** Train a Skip-Gram Word2Vec model using the same corpus. Retrieve the vector for `language` and display the five most similar words.

In [ ]:
# Skip-Gram: sg=1
skipgram_model = Word2Vec(
    sentences=tokenized_corpus,
    vector_size=50,
    window=3,
    min_count=1,
    workers=1,
    sg=1,
    seed=42,
    epochs=100
)

selected_word = "language"
print(f"Vector representation of '{selected_word}':")
print(skipgram_model.wv[selected_word])

print("\nFive most similar words:")
for word, similarity in skipgram_model.wv.most_similar(selected_word, topn=5):
    print(f"{word}: {similarity:.4f}")

**Observation:** Skip-Gram predicts surrounding context words from a target word. Its learned vectors and nearest neighbors may differ from CBOW because the training objective is different.

# 5.7 Cosine Similarity Between Word Pairs

**Question:** Train a Word2Vec model on the given corpus and calculate cosine similarity for these pairs:
1. `machine` and `learning`
2. `natural` and `language`
3. `deep` and `learning`

In [ ]:
# Use a Word2Vec model trained on the same corpus.
similarity_model = Word2Vec(
    sentences=tokenized_corpus,
    vector_size=50,
    window=3,
    min_count=1,
    workers=1,
    sg=0,
    seed=42,
    epochs=100
)

word_pairs = [
    ("machine", "learning"),
    ("natural", "language"),
    ("deep", "learning")
]

similarity_rows = []
for word1, word2 in word_pairs:
    score = similarity_model.wv.similarity(word1, word2)
    similarity_rows.append({
        "Word 1": word1,
        "Word 2": word2,
        "Cosine Similarity": round(float(score), 4)
    })

similarity_df = pd.DataFrame(similarity_rows)
display(similarity_df)

## Result

The experiment demonstrates:
- BPE and SentencePiece tokenization using pretrained and locally trained tokenizers, with token IDs.
- POS tagging using spaCy and NLTK, including tag descriptions.
- Token frequency analysis from the supplied text corpus.
- Word2Vec CBOW and Skip-Gram training, vector extraction, and nearest-word retrieval.
- Cosine similarity calculation for the selected word pairs.

## Conclusion

Subword tokenization represents text using smaller units and maps them to numerical IDs. POS tagging identifies grammatical categories of tokens, while Word2Vec learns dense word vectors from context. CBOW and Skip-Gram use different prediction objectives, and cosine similarity provides a way to compare the learned word vectors. Results from the small training corpus are illustrative and may vary slightly across environments.

## Notes for Submission
- Run every cell in Colab before submitting if your instructor requires outputs to be saved.
- Keep the section headings and displayed tables as the lab record.
- The pretrained-tokenizer sections require internet access in Colab on the first run.